Dataset exploration exercise 1

1. Libary and device setup

In [24]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix, classification_report

In [7]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [22]:
from collections import Counter

train_labels = [label for _, label in train_dataset.samples]
valid_labels = [label for _, label in valid_dataset.samples]

train_counts = Counter(train_labels)
valid_counts = Counter(valid_labels)

print("Training set class counts:")
for class_name, class_idx in train_dataset.class_to_idx.items():
    print(f"{class_name}: {train_counts[class_idx]}")

print("\nValidation set class counts:")
for class_name, class_idx in valid_dataset.class_to_idx.items():
    print(f"{class_name}: {valid_counts[class_idx]}")

Training set class counts:
fake: 219470
real: 42690

Validation set class counts:
fake: 1524
real: 1548


2. Converts images to tensors, loads training and validation images from folder names as class labels, and creates batches for model training and validation.

In [25]:
transform = transforms.Compose([
    transforms.ToTensor()
])

# Use subfolders as labels
train_dataset = datasets.ImageFolder(
    root="/Users/htetaunglwin/Desktop/Deepfake_detection/dataset/train",
    transform=transform
)

# Use subfolders as labels
valid_dataset = datasets.ImageFolder(
    root="/Users/htetaunglwin/Desktop/Deepfake_detection/dataset/valid",
    transform=transform
)

print("Train classes:", train_dataset.classes)
print("Train mapping:", train_dataset.class_to_idx)

print("Valid classes:", valid_dataset.classes)
print("Valid mapping:", valid_dataset.class_to_idx)

# Get indices for fake and real images in training set
fake_indices = [i for i, (_, label) in enumerate(train_dataset.samples) if label == train_dataset.class_to_idx['fake']]
real_indices = [i for i, (_, label) in enumerate(train_dataset.samples) if label == train_dataset.class_to_idx['real']]

# Randomly select 10000 fake and 10000 real images
selected_fake = random.sample(fake_indices, 10000)
selected_real = random.sample(real_indices, 10000)

# Combine and shuffle selected indices
selected_indices = selected_fake + selected_real
random.shuffle(selected_indices)

# Create balanced training subset
train_subset = Subset(train_dataset, selected_indices)

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True) #mini-batch
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

print("Subset training size:", len(train_subset))
print("Number of training batches:", len(train_loader))

Train classes: ['fake', 'real']
Train mapping: {'fake': 0, 'real': 1}
Valid classes: ['fake', 'real']
Valid mapping: {'fake': 0, 'real': 1}
Subset training size: 20000
Number of training batches: 625


In [26]:
# Get one batch of training images and labels
images, labels = next(iter(train_loader))

# Print batch shape
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First 10 labels:", labels[:10])

Image batch shape: torch.Size([32, 3, 256, 256])
Label batch shape: torch.Size([32])
First 10 labels: tensor([1, 1, 0, 1, 0, 0, 0, 0, 1, 1])


In [27]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc = nn.Linear(32 * 128 * 128, 2)  # for 256 x 256 input

    def forward(self, x):
        x = self.conv1(x)   # convolution
        x = self.relu(x)    # activation
        x = self.pool(x)    # pooling
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc(x)      # final classification
        return x

In [28]:
# Create the model
model = SimpleCNN().to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=524288, out_features=2, bias=True)
)


In [31]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct / total
    avg_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")

Epoch [1/5], Loss: 0.3138, Validation Accuracy: 61.82%
Epoch [2/5], Loss: 0.2571, Validation Accuracy: 63.80%
Epoch [3/5], Loss: 0.2129, Validation Accuracy: 62.34%
Epoch [4/5], Loss: 0.1759, Validation Accuracy: 64.55%
Epoch [5/5], Loss: 0.1431, Validation Accuracy: 65.59%
